# Notebook 07: Ray Tune

## Why This Matters

Hyperparameter tuning is one of the highest-ROI activities in ML engineering. The difference between a poorly-tuned and well-tuned model can be 5-20% in accuracy. Manual grid search scales O(n^k) with n values per parameter and k parameters -- completely impractical. Ray Tune provides population-based search algorithms (ASHA, PBT) that find good configurations 10-100x more efficiently than grid search. It is used at scale by Ant Group, Shopify, and many others for continuous HPO in production.


In [ ]:
%pip install -q 'ray[tune]' optuna torch numpy matplotlib

In [ ]:
import ray
from ray import tune
from ray.tune import Tuner, TuneConfig
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import ASHAScheduler, PopulationBasedTraining
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time

if ray.is_initialized():
    ray.shutdown()
ray.init(num_cpus=4, ignore_reinit_error=True)
print("Ready.")

## 1. Why Grid Search is Dead

Grid search over k hyperparameters with n values each requires $n^k$ trials. For:
- learning_rate: 5 values
- batch_size: 4 values
- hidden_size: 4 values
- dropout: 3 values
- weight_decay: 3 values

That is $5 \times 4 \times 4 \times 3 \times 3 = 720$ trials. At 10 minutes each: 120 GPU-hours.

Modern approaches:
- **Random search**: sample uniformly from the space. 10x more efficient than grid for high-dimensional spaces (Bergstra & Bengio, 2012)
- **Bayesian optimization (BO)**: build a surrogate model of the objective, sample where expected improvement is highest
- **ASHA**: early stopping -- kill bad configurations early, continue promising ones
- **PBT**: exploit successful configs by copying and perturbing their hyperparameters


In [ ]:
# Tune search spaces
print('Ray Tune search space examples:')
print(''
      'config = {')
print('  lr: tune.loguniform(1e-5, 1e-1),  # log-uniform: good for lr')
print('  batch_size: tune.choice([32, 64, 128, 256]),  # categorical')
print('  hidden: tune.randint(64, 512),    # integer range')
print('  dropout: tune.uniform(0.0, 0.5),  # continuous uniform')
print('  wd: tune.loguniform(1e-6, 1e-2),  # log-scale weight decay')
print('  n_layers: tune.grid_search([2, 3, 4]),  # exhaustive grid')
print('}')

print('\nKey distributions:')
print('  loguniform: use for lr, weight_decay -- effects are multiplicative')
print('  uniform: use for dropout rate, momentum -- effects are additive')
print('  choice: use for discrete categorical (activation, optimizer)')
print('  randint: use for integer hyperparams (n_layers, n_heads)')

## 2. ASHA: Async Successive Halving

ASHA (Asynchronous Successive Halving Algorithm) is the most commonly used scheduler for HPO:

**How it works**:
1. Start many trials with random hyperparameters
2. After `min_t` epochs, evaluate all running trials
3. Keep only the top 1/eta fraction, discard the rest
4. Double the resource budget and repeat

The key: **kill bad trials early**, before they waste compute. A trial that is in the bottom 50% after 5 epochs is almost certainly going to stay there.

**Reduction factor eta**: typically 3 or 4. With eta=3 and max_t=81:
- Rung 0 (t=3): 27 trials survive to this rung (start with 81 * eta^3 = 2187 trials)
- Rung 1 (t=9): 9 survive
- Rung 2 (t=27): 3 survive
- Rung 3 (t=81): 1 survives

Total compute: 2187*3 + 729*9 + 243*27 + 81*81 ≈ 39K vs 2187*81 ≈ 177K for full training. 4.5x savings.


In [ ]:
# ASHA: early stopping HPO

def train_fn_asha(config):
    import numpy as np
    import torch
    import torch.nn as nn
    
    torch.manual_seed(0)
    np.random.seed(0)
    
    # Synthetic dataset
    N = 500
    X = torch.randn(N, 16)
    y = (X[:, 0] + X[:, 1] * X[:, 2] > 0).long()
    
    model = nn.Sequential(
        nn.Linear(16, config['hidden']),
        nn.ReLU(),
        nn.Dropout(config['dropout']),
        nn.Linear(config['hidden'], 2)
    )
    optimizer = torch.optim.Adam(model.parameters(),
                                  lr=config['lr'],
                                  weight_decay=config.get('wd', 0))
    
    for epoch in range(20):
        model.train()
        logits = model(X)
        loss = nn.functional.cross_entropy(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_acc = (model(X).argmax(1) == y).float().mean().item()
        
        # Report every epoch -- ASHA scheduler uses this to decide early stopping
        tune.report({'loss': loss.item(), 'accuracy': val_acc})

# ASHA scheduler: kill bottom 50% every 2 epochs
asha_scheduler = ASHAScheduler(
    metric='accuracy',
    mode='max',
    max_t=20,          # max epochs
    grace_period=3,    # min epochs before any trial is stopped
    reduction_factor=2 # keep top 1/2 each round
)

# Optuna Bayesian search
optuna_search = OptunaSearch(metric='accuracy', mode='max')

tuner = Tuner(
    train_fn_asha,
    param_space={
        'lr':      tune.loguniform(1e-4, 1e-1),
        'hidden':  tune.choice([16, 32, 64, 128]),
        'dropout': tune.uniform(0.0, 0.5),
        'wd':      tune.loguniform(1e-6, 1e-3),
    },
    tune_config=TuneConfig(
        num_samples=20,      # total trials to start
        scheduler=asha_scheduler,
        search_alg=optuna_search,
    ),
)

results = tuner.fit()
best = results.get_best_result(metric='accuracy', mode='max')
print(f'Best accuracy: {best.metrics["accuracy"]:.4f}')
print(f'Best config:   {best.config}')

## 3. PBT: Population-Based Training

PBT (Jaderberg et al. 2017, DeepMind) combines HPO with asynchronous training:

**How it works**:
1. Start a **population** of workers with different hyperparameters
2. Periodically, evaluate each worker's performance
3. **Exploit**: copy weights from a top-performing worker to a poor-performing worker
4. **Explore**: randomly perturb the copied worker's hyperparameters

Key insight: PBT does not just find good hyperparameters -- it finds a good **schedule** of hyperparameters. Learning rate warmup/decay emerges automatically.

PBT is particularly powerful for reinforcement learning (where DeepMind first introduced it) and long training runs where hyperparameters should change over time.


In [ ]:
# Plot: ASHA efficiency visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Simulate ASHA elimination rounds
np.random.seed(42)
n_initial = 50
max_epochs = 20
grace = 3

# Each trial's final performance correlates with early performance + noise
true_quality = np.random.beta(2, 5, n_initial)  # most are bad, few are good

# ASHA: kill bottom 50% at rounds
ax = axes[0]
rounds = [3, 6, 12, 20]
surviving_indices = list(range(n_initial))
colors = plt.cm.Reds(np.linspace(0.3, 1.0, n_initial))
sort_idx = np.argsort(true_quality)

y_pos = np.arange(n_initial)
ax.barh(y_pos, true_quality[sort_idx], color='steelblue', alpha=0.6)
ax.set_xlabel('Final accuracy (if fully trained)')
ax.set_ylabel('Trial rank')
ax.set_title(f'ASHA: starts {n_initial} trials, kills bottom 50% repeatedly')
ax.axhline(y=n_initial//2, color='red', linestyle='--', label='ASHA cutoff at round 1')
ax.axhline(y=n_initial*3//4, color='orange', linestyle='--', label='ASHA cutoff at round 2')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Learning curve comparison: grid search vs ASHA
ax = axes[1]
epochs = np.arange(1, max_epochs+1)

# Grid search: all trials run to completion
grid_compute = n_initial * epochs

# ASHA: half eliminated at each round
asha_surviving = n_initial
asha_compute = np.zeros(max_epochs)
for e in range(max_epochs):
    asha_compute[e] = asha_surviving
    if e in [grace-1, 2*grace-1, 4*grace-1]:
        asha_surviving = max(1, asha_surviving // 2)
asha_cumulative = np.cumsum(asha_compute)

ax.plot(epochs, grid_compute, 'r-o', markersize=4, label='Grid search')
ax.plot(epochs, asha_cumulative, 'g-o', markersize=4, label='ASHA')
ax.set_xlabel('Epoch')
ax.set_ylabel('Total compute (trials * epochs)')
ax.set_title('Cumulative compute: Grid Search vs ASHA')
ax.legend()
ax.grid(True, alpha=0.3)

savings = grid_compute[-1] / asha_cumulative[-1]
ax.text(max_epochs*0.6, grid_compute[-1]*0.5, f'{savings:.1f}x savings',
       fontsize=12, color='green', fontweight='bold')

plt.tight_layout()
plt.savefig('asha_efficiency.png', dpi=100)
plt.show()
print(f'ASHA saves {savings:.1f}x compute vs full grid search')

## Summary

### Key Concepts

| Algorithm | Type | Best for | Key parameter |
|-----------|------|---------|---------------|
| Random search | Exploration | High-dimensional spaces | `num_samples` |
| Bayesian (Optuna/HyperOpt) | Model-based | Low-dimensional, expensive eval | `num_samples` |
| ASHA | Early stopping | Many trials, long training | `grace_period`, `reduction_factor` |
| PBT | Exploit+explore | Long training, adaptive schedules | `perturbation_interval` |

**Key insight**: ASHA's power is asymmetric -- it is cheap to run many short trials but expensive to run few long ones. ASHA allocates resources optimally by eliminating poor performers early.

**Next**: `08_ray_data.ipynb` -- scalable data loading for pre-training scale.
